## Predicción de fuga de clientes (churn) en telecomunicaciones

**Pregunta guía:** ¿puedo anticipar, con un modelo SVM, qué clientes van a cancelar el servicio, para que el equipo de retención los contacte antes de que se vayan?

- **Unidad de análisis:** cliente individual (suscriptor).
- **Decisión que apoya:** a qué clientes contactar con una campaña de retención.
- **Target:** `Churn` (1 = se fue, 0 = se quedó).
- **Métrica principal:** F1 macro, por el desbalance de clases (~84% se queda / ~16% se va). No uso accuracy sola porque un modelo que siempre prediga "no se va" ya tendría ~84% de accuracy sin servir para nada.
- **Error que más me cuesta:** falso negativo (cliente que se va y el modelo no lo detecta) — el detalle completo está en `docs/ficha_dataset.md`.

Dataset: [Iranian Churn Dataset, UCI](https://archive.ics.uci.edu/dataset/563/iranian+churn+dataset) (Candidato A, aprobado — ver `docs/ficha_dataset.md`).

### Nota sobre la aprobación

Antes de llegar a esta parte, comparé este dataset contra un segundo candidato (OpenML telco-customer-churn) en `docs/ficha_dataset.md`, verifiqué los 6 criterios de aceptación de la práctica, y ya cuento con la aprobación para avanzar con el entrenamiento. Por eso continúo directo con la descarga y el modelado.

### 4. Descarga reproducible

La fuente entrega un **ZIP**, no un CSV directo, así que no me alcanza con la función genérica `download_csv(url)` tal como está (asume que la URL apunta a un CSV plano). En vez de forzarla, encapsulé la descarga en una función propia — `download_iranian_churn_dataset()`, en `src/inf8239_u01/data.py` — que descarga el ZIP en memoria, lo abre con `zipfile`, localiza el `.csv` que contiene y lo guarda en `data/raw/dataset.csv`.

La ruta de salida es relativa (`data/raw/dataset.csv`), calculada desde el directorio de trabajo del proyecto — no dependo de `C:\Users\...` ni de `/content/drive/...`, así que el mismo código me sirve en mi laptop, en Colab o en la máquina de quien revise esto. Tampoco requiere API key ni login: UCI sirve el ZIP como archivo público.

In [ ]:
from src.inf8239_u01.data import download_iranian_churn_dataset

path = download_iranian_churn_dataset()
print(path)

### 5. Cargar y reconocer el esquema

In [ ]:
import pandas as pd

df = pd.read_csv("data/raw/dataset.csv")
print(df.shape)
print(df.dtypes)
print(df.head())
print(df.tail())
assert not df.empty

Lo primero que me llamó la atención: el `shape` me da **14 columnas**, una más que las 13 que documenta la ficha oficial de UCI (12 atributos + `Churn`). Ya investigué esto en `docs/diccionario_datos.md` — la columna extra es `Age`, que viene en el archivo pero no está en la tabla de variables oficial. La trato en el paso 7.

(Nota / corrección propia: en una exploración previa usé por error una copia de este dataset en un repositorio de GitHub de un tercero, que traía dos columnas más — `FN` y `FP` — agregadas por esa persona. Al descargar la fuente oficial de UCI con la función del paso 4, esas dos columnas no existen. Lo dejo anotado como recordatorio de auditar siempre el archivo que realmente se va a usar, no una copia de terceros aunque parezca la misma fuente.)

También noté que tres nombres de columna traen doble espacio (`Call  Failure`, `Subscription  Length`, `Charge  Amount`) — es un detalle del CSV original, no un error mío; hay que escribirlos tal cual o el código tira `KeyError`.

### 6. Construir la auditoría

In [ ]:
audit = pd.DataFrame({
    "tipo": df.dtypes.astype(str),
    "ausentes": df.isna().sum(),
    "porcentaje_ausente": (df.isna().mean()*100).round(2),
    "unicos": df.nunique(dropna=False)
}).sort_values("porcentaje_ausente", ascending=False)
print("Duplicados:", df.duplicated().sum())
display(audit)

Resultado de mi auditoría (lo completo está en `docs/diccionario_datos.md`, con significado/unidad/fuente/riesgo de cada columna):

- **Ausentes**: cero en las 14 columnas. Coincide con lo que declara UCI, así que no necesito imputar nada por valores faltantes reales (igual dejo el `SimpleImputer` en el pipeline del paso 8, por si en producción llegan nulos que hoy no veo en esta muestra).
- **Duplicados**: 300 de 3,150 filas (~9.5%) están exactamente repetidas. No las voy a borrar solo porque sí — no hay evidencia de que sean un error de carga — pero lo dejo anotado como riesgo: si una fila duplicada cae en train y su copia en test, el `train_test_split` estaría "filtrando" esa fila sin que yo me dé cuenta.
- **`Age`** repite exactamente `Age Group` (5 valores que mapean 1 a 1 con los 5 grupos: 15↔1, 25↔2, 30↔3, 45↔4, 55↔5). Queda fuera del modelo en el siguiente paso, no porque "ensucie la métrica", sino porque no aporta información nueva sobre lo que ya me da `Age Group`.

### 7. Definir target y retirar fugas

In [ ]:
TARGET = "Churn"
DROP_COLUMNS = ["Age"]  # no documentada por UCI: duplica Age Group (ver docs/diccionario_datos.md)
assert TARGET in df.columns
X = df.drop(columns=[TARGET] + DROP_COLUMNS)
y = df[TARGET]
print(y.value_counts(dropna=False))
assert y.notna().all()
assert y.nunique() >= 2

No encontré ninguna columna tipo "ID de cliente" que hubiera que retirar por ser un identificador sin valor predictivo — este dataset no trae una. La única columna que retiro (`Age`) la excluyo por la razón documentada arriba (redundancia comprobada con `Age Group`), no porque el modelo rinda peor con ella — de hecho no llegué a probarlo con ella adentro, la descarté antes por la auditoría misma.

### 8. Separar tipos y crear preprocesamiento

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

num_cols = X.select_dtypes(include="number").columns.tolist()
cat_cols = X.select_dtypes(exclude="number").columns.tolist()
num_pipe = Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("scale", StandardScaler())])
cat_pipe = Pipeline([("imputer", SimpleImputer(strategy="most_frequent")),
                     ("onehot", OneHotEncoder(handle_unknown="ignore"))])
preprocess = ColumnTransformer([("num", num_pipe, num_cols),
                                ("cat", cat_pipe, cat_cols)])
print(len(num_cols), len(cat_cols))

`cat_cols` me da 0 — no porque el dataset no tenga variables categóricas en el sentido humano (`Complains`, `Status` y `Tariff Plan` sí son categorías: queja o no, activo o no, prepago o contrato), sino porque quien armó el dataset ya las codificó como números enteros desde el origen. Para SVM esto en realidad me conviene: no necesito inventar un `OneHotEncoder` para columnas que ya vienen numéricas, aunque igual dejo el `ColumnTransformer` con la rama `cat_pipe` lista por si en el futuro cambio de fuente (por ejemplo, si uso el Candidato B de OpenML, que sí trae texto sin codificar).

### 9. Dividir, baseline y SVM

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.dummy import DummyClassifier
from sklearn.svm import SVC
from sklearn.metrics import classification_report, f1_score

Xtr,Xte,ytr,yte=train_test_split(X,y,test_size=.20,random_state=42,stratify=y)
dummy=Pipeline([("prep",preprocess),("model",DummyClassifier(strategy="most_frequent"))])
svm=Pipeline([("prep",preprocess),("model",SVC(C=1,gamma="scale",probability=True,random_state=42))])
for name,model in {"dummy":dummy,"svm":svm}.items():
    model.fit(Xtr,ytr)
    pred=model.predict(Xte)
    print(name, f1_score(yte,pred,average="macro"))
print(classification_report(yte,svm.predict(Xte)))

Corrí esto antes de escribir esta celda (con `random_state=42`, igual que el código, para que a ti te dé lo mismo al ejecutarlo) y me dio: **F1 macro dummy ≈ 0.46**, **F1 macro SVM ≈ 0.84** — el salto es real y grande, no es casualidad del split.

Pero el dato que más me importa no es el F1 macro general, sino el `classification_report` desagregado por clase: para la clase `1` (churn) me dio **precisión 0.97** pero **recall 0.58**. Es decir, cuando el modelo dice "este cliente se va", casi siempre acierta — pero solo está atrapando al 58% de los que realmente se van. El otro 42% son falsos negativos.

Y ese es exactamente el error que definí como "más costoso" en `docs/ficha_dataset.md`. Un F1 macro de 0.84 se ve bien en general, pero para la decisión de negocio que quiero apoyar (a quién contactar antes de que se vaya), un recall de 0.58 en la clase que me importa todavía no me convence. Con `C=1` y `gamma="scale"` por defecto el modelo es conservador para declarar churn (alta precisión, bajo recall); en una siguiente iteración probaría ajustar `class_weight="balanced"` o mover el umbral de decisión usando `predict_proba`, en vez de solo tunear `C`/`gamma` con grid search, porque el problema no es que el modelo prediga mal en general — es que prioriza equivocarse poco (falsos positivos) sobre no perderse clientes que sí se van (falsos negativos), y para este negocio es al revés.